<a href="https://colab.research.google.com/github/tabrejansary/ML-Assignments/blob/main/Lab-06/A1-A9_Repeated_using_AI_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

import os
import glob
import numpy as np
import pandas as pd

from PIL import Image
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

Mounted at /content/drive


In [2]:
DRIVE_DIR = "/content/drive/MyDrive"

DATASET_DIR = os.path.join(
    DRIVE_DIR,
    "BreaKHis_v1"
)

CSV_PATH = os.path.join(
    DRIVE_DIR,
    "BreaKHis_metadata.csv"
)

SAMPLES_PER_CLASS = 100

IMAGE_SIZE = (16, 16)

K = 3

DISTANCE_METRIC = "euclidean"

SORT_ALGORITHM = "insertion"

RANDOM_STATE = 42

In [3]:
data = pd.read_csv(CSV_PATH)

print("Loaded:", CSV_PATH)

print("Columns:", data.columns.tolist())

print(data.head())

Loaded: /content/drive/MyDrive/BreaKHis_metadata.csv
Columns: ['fold', 'mag', 'grp', 'filename']
   fold  mag    grp                                           filename
0     1  100  train  BreaKHis_v1/histology_slides/breast/benign/SOB...
1     1  100  train  BreaKHis_v1/histology_slides/breast/benign/SOB...
2     1  100  train  BreaKHis_v1/histology_slides/breast/benign/SOB...
3     1  100  train  BreaKHis_v1/histology_slides/breast/benign/SOB...
4     1  100  train  BreaKHis_v1/histology_slides/breast/benign/SOB...


In [ ]:
# A1
# GenAI Tool Used: ChatGPT

def convert_categories(data):
    """
    Convert categorical columns into numerical representations.
    """
    result = data.copy()

    categorical_cols = result.select_dtypes(
        include=["object", "category"]
    ).columns

    for col in categorical_cols:
        unique_values = result[col].dropna().unique()

        value_map = {
            value: number
            for number, value in enumerate(unique_values)
        }

        result[col] = result[col].map(value_map)

    return result


def fill_missing_data(data, technique="mean"):
    """
    Replace missing values using mean, median, or mode.
    """
    result = data.copy()

    for col in result.columns:
        if result[col].isnull().any():

            if technique == "mean":
                replacement_value = result[col].mean()

            elif technique == "median":
                replacement_value = result[col].median()

            elif technique == "mode":
                replacement_value = result[col].mode()[0]

            else:
                raise ValueError(
                    "Technique must be mean, median, or mode."
                )

            result[col] = result[col].fillna(replacement_value)

    return result


def get_distance(vector_a, vector_b, distance_type="euclidean"):
    """
    Calculate distance between two feature vectors.
    """
    vector_a = np.asarray(vector_a, dtype=float)
    vector_b = np.asarray(vector_b, dtype=float)

    if distance_type == "euclidean":
        return np.sqrt(
            np.sum((vector_a - vector_b) ** 2)
        )

    elif distance_type == "manhattan":
        return np.sum(
            np.abs(vector_a - vector_b)
        )

    else:
        raise ValueError(
            "Distance type must be euclidean or manhattan."
        )


def bubble_neighbor_sort(neighbors):
    """
    Sort neighbors using Bubble Sort.
    """
    result = neighbors.copy()

    for i in range(len(result)):
        for j in range(len(result) - i - 1):

            first_key = (
                result[j][0],
                result[j][2]
            )

            second_key = (
                result[j + 1][0],
                result[j + 1][2]
            )

            if first_key > second_key:
                result[j], result[j + 1] = (
                    result[j + 1],
                    result[j]
                )

    return result


def selection_neighbor_sort(neighbors):
    """
    Sort neighbors using Selection Sort.
    """
    result = neighbors.copy()

    for i in range(len(result)):

        smallest = i

        for j in range(i + 1, len(result)):

            current_key = (
                result[j][0],
                result[j][2]
            )

            smallest_key = (
                result[smallest][0],
                result[smallest][2]
            )

            if current_key < smallest_key:
                smallest = j

        result[i], result[smallest] = (
            result[smallest],
            result[i]
        )

    return result


def insertion_neighbor_sort(neighbors):
    """
    Sort neighbors using Insertion Sort.
    """
    result = neighbors.copy()

    for i in range(1, len(result)):

        selected = result[i]
        position = i - 1

        selected_key = (
            selected[0],
            selected[2]
        )

        while position >= 0:

            previous_key = (
                result[position][0],
                result[position][2]
            )

            if previous_key <= selected_key:
                break

            result[position + 1] = result[position]
            position -= 1

        result[position + 1] = selected

    return result


def arrange_neighbors(neighbors, method="insertion"):
    """
    Select and apply the requested sorting algorithm.
    """
    if method == "bubble":
        return bubble_neighbor_sort(neighbors)

    elif method == "selection":
        return selection_neighbor_sort(neighbors)

    elif method == "insertion":
        return insertion_neighbor_sort(neighbors)

    else:
        raise ValueError(
            "Method must be bubble, selection, or insertion."
        )


def select_nearest(sorted_neighbors, k):
    """
    Return the k closest neighbors.
    """
    if k <= 0:
        raise ValueError("k must be greater than zero.")

    return sorted_neighbors[:min(k, len(sorted_neighbors))]


def majority_vote(neighbors):
    """
    Determine the class using majority voting.
    The closest neighbor resolves a tie.
    """
    votes = Counter()

    for distance, label, index in neighbors:
        votes[label] += 1

    highest_votes = max(votes.values())

    winning_labels = [
        label
        for label, count in votes.items()
        if count == highest_votes
    ]

    if len(winning_labels) == 1:
        return winning_labels[0]

    for distance, label, index in neighbors:
        if label in winning_labels:
            return label


def predict_knn(
    X_train,
    y_train,
    X_test,
    k=3,
    distance_type="euclidean",
    sorting_method="insertion"
):
    """
    Predict class labels using the custom kNN implementation.
    """
    predictions = []

    for test_vector in X_test:

        neighbor_data = []

        for index in range(len(X_train)):

            distance = get_distance(
                test_vector,
                X_train[index],
                distance_type
            )

            neighbor_data.append(
                (
                    distance,
                    y_train[index],
                    index
                )
            )

        ordered_neighbors = arrange_neighbors(
            neighbor_data,
            sorting_method
        )

        nearest_neighbors = select_nearest(
            ordered_neighbors,
            k
        )

        predicted_label = majority_vote(
            nearest_neighbors
        )

        predictions.append(predicted_label)

    return np.array(predictions)

In [ ]:
# A2
# GenAI Tool Used: ChatGPT

def weighted_vote(neighbors):
    """
    Assign a class based on distance-weighted voting.
    Closer neighbors receive greater importance.
    """
    weights = {}

    for distance, label, index in neighbors:

        if distance == 0:
            weight = float("inf")
        else:
            weight = 1 / distance

        if label not in weights:
            weights[label] = 0

        weights[label] += weight

    best_label = None
    best_weight = -1

    for label, weight in weights.items():

        if weight > best_weight:
            best_weight = weight
            best_label = label

    return best_label


def predict_weighted_knn(
    X_train,
    y_train,
    X_test,
    k=3,
    distance_type="euclidean",
    sorting_method="insertion"
):
    """
    Predict class labels using weighted kNN.
    """
    predictions = []

    for test_vector in X_test:

        neighbors = []

        for train_index in range(len(X_train)):

            distance = get_distance(
                test_vector,
                X_train[train_index],
                distance_type
            )

            neighbors.append(
                (
                    distance,
                    y_train[train_index],
                    train_index
                )
            )

        neighbors = arrange_neighbors(
            neighbors,
            sorting_method
        )

        nearest = select_nearest(
            neighbors,
            k
        )

        predicted_label = weighted_vote(
            nearest
        )

        predictions.append(predicted_label)

    return np.array(predictions)

In [ ]:
# A3
# GenAI Tool Used: ChatGPT

def create_train_test_sets(features, labels, test_ratio=0.3,
                           seed=42):
    """
    Divide the dataset into training and testing subsets.
    """
    train_features, test_features, train_labels, test_labels = (
        train_test_split(
            features,
            labels,
            test_size=test_ratio,
            random_state=seed,
            stratify=labels
        )
    )

    return (
        train_features,
        test_features,
        train_labels,
        test_labels
    )

In [ ]:
# A4
# GenAI Tool Used: ChatGPT

def build_knn_model(train_data, train_labels, neighbors=3):
    """
    Create and train a kNN classifier using the training dataset.
    """
    classifier = KNeighborsClassifier(
        n_neighbors=neighbors
    )

    classifier.fit(
        train_data,
        train_labels
    )

    return classifier

In [ ]:
# A5
# GenAI Tool Used: ChatGPT

def evaluate_knn_accuracy(model, test_data, test_labels):
    """
    Calculate the accuracy of the trained kNN classifier
    using the test dataset.
    """
    accuracy = model.score(
        test_data,
        test_labels
    )

    return accuracy


accuracy = evaluate_knn_accuracy(
    knn_model,
    X_test,
    y_test
)

print("\nA5")
print("kNN Accuracy:", round(accuracy, 4))